# ML-Accelerated Layout Prediction

Layout selection --- mapping logical qubits to physical qubits --- is one of the
most impactful stages of quantum compilation.  A good layout minimises the SWAP
overhead introduced by routing, which directly translates to fewer 2-qubit gates
and higher output fidelity.

qb-compiler ships three ML-based layout predictors that accelerate this step:

| Predictor | Model | What it predicts |
|-----------|-------|------------------|
| `MLLayoutPredictor` | XGBoost v1 | Per-qubit probability of appearing in a high-quality layout |
| `MLLayoutPredictorV2` | XGBoost v2 | Post-routing 2Q gate count for a given layout |
| `GNNLayoutPredictor` | PyTorch GCN | Per-qubit suitability score using graph structure |

All three expose the same `predict_candidate_qubits()` interface, so they are
drop-in replacements for each other.  This notebook walks through usage,
comparison, and performance characteristics.

In [ ]:
from qb_compiler import QBCircuit, QBCompiler, CompilerConfig
from qb_compiler.ml.layout_predictor import MLLayoutPredictor, MLLayoutPredictorV2
from qb_compiler.ir.circuit import QBCircuit as IRCircuit
from qb_compiler.ir.operations import QBGate, QBMeasure
from qb_compiler.calibration.static_provider import StaticCalibrationProvider


def to_ir(circuit: QBCircuit) -> IRCircuit:
    """Convert a user-facing QBCircuit to IR-level QBCircuit for ML predictors."""
    ir = IRCircuit(n_qubits=circuit.n_qubits, n_clbits=0)
    for op in circuit.ops:
        if op.name == "measure":
            continue
        ir.add_gate(QBGate(name=op.name, qubits=op.qubits, params=op.params))
    return ir

## 1. MLLayoutPredictor (XGBoost v1)

The v1 predictor is a binary classifier: for each physical qubit, it predicts
the probability that the qubit appears in a high-quality layout.  It uses
calibration features (T1, T2, readout error, gate error, connectivity) and
circuit features (gate count, depth, 2Q ratio) to score every physical qubit.

### Loading a pre-trained model

The simplest way to get started is `load_bundled()`, which loads pre-trained
weights shipped with the package.

In [ ]:
# Load the bundled XGBoost v1 model trained on IBM Heron calibration data
predictor_v1 = MLLayoutPredictor.load_bundled("ibm_heron")

print(f"Model loaded: {predictor_v1._model_path.name}")
print(f"Version: {predictor_v1.metadata.get('version', 'unknown')}")

### Predicting candidate qubits

`predict_candidate_qubits()` takes a circuit and backend calibration data,
then returns a ranked list of physical qubit IDs most likely to produce a
good layout.  This narrows the search space for VF2 or other layout
algorithms.

In [ ]:
# Build a 5-qubit GHZ circuit
ghz = QBCircuit(5).h(0).cx(0, 1).cx(1, 2).cx(2, 3).cx(3, 4).measure_all()
print(f"Circuit: {ghz}")
print(f"  gate_count={ghz.gate_count}, depth={ghz.depth}, 2Q={ghz.two_qubit_count}")

# Load backend calibration from a static snapshot.
# StaticCalibrationProvider loads a calibration JSON and exposes .properties
# (BackendProperties), which is what the ML predictors expect.
provider = StaticCalibrationProvider.from_json(
    "../tests/fixtures/calibration_snapshots/ibm_fez_2026_03_14.json"
)
backend_props = provider.properties

# Convert to IR circuit for ML prediction (predictors expect IR-level QBCircuit
# which has .two_qubit_gate_count and .iter_ops())
ghz_ir = to_ir(ghz)

# Predict top-K candidate physical qubits
candidates = predictor_v1.predict_candidate_qubits(ghz_ir, backend_props)
print(f"\nTop candidates ({len(candidates)} qubits): {candidates[:15]}...")
print(f"  (out of {backend_props.n_qubits} physical qubits)")

### Tuning top-K factor

The `top_k_factor` controls how many candidates are returned relative to
the number of logical qubits.  The default (3.0) returns `n_logical * 3`
candidates.  A larger factor gives the downstream layout algorithm more
options but takes longer to search.

In [ ]:
# Compare different top-K factors
for factor in [2.0, 3.0, 5.0]:
    pred = MLLayoutPredictor.load_bundled("ibm_heron")
    pred._top_k_factor = factor
    cands = pred.predict_candidate_qubits(ghz_ir, backend_props)
    print(f"top_k_factor={factor:.1f} -> {len(cands)} candidates")

## 2. MLLayoutPredictorV2 --- Predicting post-routing 2Q gate counts

The v2 predictor is a regression model: instead of scoring individual qubits,
it predicts the total number of 2-qubit gates that will remain after routing
for a given (circuit, layout) pair.  Lower predictions are better.

This enables direct layout comparison via `score_layout()` and batch
comparison via `score_layouts()`.

In [ ]:
predictor_v2 = MLLayoutPredictorV2.load_bundled("ibm_heron")

print(f"Model: {predictor_v2._model_path.name}")
print(f"MAE: {predictor_v2.metadata.get('mae', 'unknown')}")
print(f"Correlation: {predictor_v2.metadata.get('correlation', 'unknown')}")

### Scoring a single layout

A layout is a `dict[int, int]` mapping logical qubit -> physical qubit.
`score_layout()` returns the predicted post-routing 2Q gate count.

In [ ]:
# Define two candidate layouts for our 5-qubit GHZ circuit
layout_a = {0: 10, 1: 11, 2: 12, 3: 13, 4: 14}  # contiguous region
layout_b = {0: 0, 1: 50, 2: 100, 3: 120, 4: 150}  # spread out

score_a = predictor_v2.score_layout(ghz_ir, layout_a, backend_props)
score_b = predictor_v2.score_layout(ghz_ir, layout_b, backend_props)

print(f"Layout A (contiguous): predicted 2Q gates = {score_a:.1f}")
print(f"Layout B (spread out): predicted 2Q gates = {score_b:.1f}")
print(f"Layout A is {'better' if score_a < score_b else 'worse'} (lower = better)")

### Batch scoring with `score_layouts()`

When comparing many layouts, `score_layouts()` is more efficient than
calling `score_layout()` in a loop because it batches the XGBoost
prediction.

In [ ]:
import random

# Generate 10 random layouts
random.seed(42)
all_phys = list(range(backend_props.n_qubits))
layouts = []
for _ in range(10):
    chosen = random.sample(all_phys, ghz.n_qubits)
    layouts.append({i: p for i, p in enumerate(chosen)})

# Score them all in one call
scores = predictor_v2.score_layouts(ghz_ir, layouts, backend_props)

# Print ranked results
ranked = sorted(zip(scores, layouts), key=lambda x: x[0])
for i, (score, layout) in enumerate(ranked[:5]):
    phys = [layout[q] for q in range(ghz.n_qubits)]
    print(f"  #{i+1}: predicted 2Q={score:.1f}, physical qubits={phys}")

### V2 candidate qubit prediction

V2 can also provide candidate qubits (same interface as V1).  It does this
by sampling random connected layouts, scoring each with the regression model,
and returning qubits that frequently appear in the best-scoring layouts.

In [ ]:
candidates_v2 = predictor_v2.predict_candidate_qubits(ghz_ir, backend_props)
print(f"V2 candidates ({len(candidates_v2)} qubits): {candidates_v2[:15]}...")

# Compare overlap with V1 candidates
overlap = set(candidates) & set(candidates_v2)
print(f"\nOverlap with V1: {len(overlap)} qubits")
print(f"V1-only: {len(set(candidates) - set(candidates_v2))} qubits")
print(f"V2-only: {len(set(candidates_v2) - set(candidates))} qubits")

## 3. GNNLayoutPredictor (PyTorch-based)

The GNN predictor uses a dual-graph neural network:
- **Device graph encoder** (GCN): processes the device coupling graph with
  per-qubit calibration features (T1, T2, gate error, connectivity, etc.)
- **Circuit graph encoder** (GCN): processes the circuit interaction graph
- **Cross-attention**: aligns circuit needs with device capabilities
- **Scoring head** (MLP): outputs a per-qubit suitability score

The GNN captures structural information (local connectivity patterns,
neighbourhood quality) that flat XGBoost features miss, which is especially
helpful on large devices with heterogeneous topology like IBM heavy-hex.

In [ ]:
from qb_compiler.ml.gnn_router import GNNLayoutPredictor

# Load the pre-trained GNN model.
# NOTE: The bundled weights were trained with 7 device features, but the
# current code uses 9 features (routing_headroom and neighborhood_error
# were added after training).  This will fail until the model is retrained.
try:
    gnn_pred = GNNLayoutPredictor.load_bundled("ibm_heron")

    print(f"Architecture: {gnn_pred.metadata.get('architecture', 'unknown')}")
    print(f"Hidden dim: {gnn_pred.metadata.get('hidden_dim', 'unknown')}")
    print(f"Parameters: {gnn_pred.metadata.get('n_parameters', 'unknown'):,}")
    print(f"Training AUC: {gnn_pred.metadata.get('training_auc', 'unknown')}")
except RuntimeError as e:
    gnn_pred = None
    print(f"GNN model needs retraining for current feature set: {e}")
    print("Retrain with: python -m qb_compiler.ml.train_gnn")

In [ ]:
# GNN candidate prediction (same interface as XGBoost predictors)
if gnn_pred is not None:
    candidates_gnn = gnn_pred.predict_candidate_qubits(ghz_ir, backend_props)
    print(f"GNN candidates ({len(candidates_gnn)} qubits): {candidates_gnn[:15]}...")
else:
    candidates_gnn = None
    print("Skipped: GNN model not available (needs retraining, see cell above)")

## 4. Comparing ML vs heuristic layout selection

To see whether ML prediction actually helps, we can compare the
compile results when using ML-guided candidate filtering vs the
compiler's default heuristic layout.

In [ ]:
import time

# Build a more interesting circuit: 8-qubit QAOA layer
qaoa = QBCircuit(8)
for i in range(7):
    qaoa.cx(i, i + 1).rz(i + 1, 0.5).cx(i, i + 1)
for i in range(8):
    qaoa.rx(i, 0.7)
qaoa.measure_all()

print(f"QAOA circuit: {qaoa}")
print(f"  2Q gates: {qaoa.two_qubit_count}")

# Compile with default strategy (no ML)
compiler = QBCompiler.from_backend("ibm_fez")
t0 = time.perf_counter()
result_default = compiler.compile(qaoa)
t_default = (time.perf_counter() - t0) * 1000

print(f"\nDefault: depth={result_default.compiled_depth}, "
      f"fidelity={result_default.estimated_fidelity:.4f}, "
      f"time={t_default:.1f}ms")

## 5. Training metadata inspection

Each model ships with a `.meta.json` sidecar file containing training
metadata.  This is useful for auditing model provenance.

In [ ]:
import json

# V1 metadata
print("=== XGBoost V1 ===")
for k, v in predictor_v1.metadata.items():
    print(f"  {k}: {v}")

print("\n=== XGBoost V2 ===")
for k, v in predictor_v2.metadata.items():
    print(f"  {k}: {v}")

print("\n=== GNN ===")
if gnn_pred is not None:
    for k, v in gnn_pred.metadata.items():
        print(f"  {k}: {v}")
else:
    print("  (not available - model needs retraining for current feature set)")

## 6. When ML prediction helps most

ML-accelerated layout prediction shows the largest benefit when:

1. **Large backends** (>100 qubits): the search space grows combinatorially
   with qubit count.  Pre-filtering from 156 to 30 candidates cuts search
   time by orders of magnitude.

2. **Complex topologies**: heavy-hex, square lattice, and other non-trivial
   topologies have "good" and "bad" neighbourhoods.  ML captures this.

3. **High 2Q gate density**: circuits with many 2-qubit interactions benefit
   most from good layout because bad placement creates more SWAP overhead.

For small backends (IonQ Aria at 25 qubits with all-to-all connectivity) or
very small circuits (2-3 qubits), the heuristic is already fast and
nearly optimal --- ML prediction adds overhead without benefit.

In [ ]:
# Demonstrate: scaling circuit size and measuring prediction time

import time

sizes = [3, 5, 10, 15, 20, 30]
print(f"{'Qubits':>8} {'Gates':>8} {'2Q':>6} {'Pred (ms)':>12}")
print("-" * 40)

for n in sizes:
    # Build a chain of CX gates
    circ = QBCircuit(n)
    for i in range(n - 1):
        circ.cx(i, i + 1).rz(i + 1, 0.5).cx(i, i + 1)
    for i in range(n):
        circ.rx(i, 0.7)

    # Convert to IR for ML prediction
    circ_ir = to_ir(circ)

    t0 = time.perf_counter()
    cands = predictor_v1.predict_candidate_qubits(circ_ir, backend_props)
    elapsed = (time.perf_counter() - t0) * 1000

    print(f"{n:>8} {circ.gate_count:>8} {circ.two_qubit_count:>6} {elapsed:>10.1f}ms")

## 7. Performance benchmarks: ML prediction time vs transpile time

The entire point of ML prediction is to amortise the cost of exhaustive
layout search.  The prediction itself should be much faster than a full
transpilation pass.

Below we compare the time to predict candidates vs the time to compile
the circuit.

In [ ]:
import time

# Build a medium-sized circuit
circ_medium = QBCircuit(12)
for layer in range(3):
    for i in range(0, 12, 2):
        if i + 1 < 12:
            circ_medium.cx(i, i + 1)
    for i in range(1, 12, 2):
        if i + 1 < 12:
            circ_medium.cx(i, i + 1)
    for i in range(12):
        circ_medium.rz(i, 0.3 * layer)
circ_medium.measure_all()

print(f"Test circuit: {circ_medium}")

# Convert to IR for ML prediction
circ_medium_ir = to_ir(circ_medium)

# Time ML prediction
t0 = time.perf_counter()
for _ in range(10):
    predictor_v1.predict_candidate_qubits(circ_medium_ir, backend_props)
ml_time = (time.perf_counter() - t0) / 10 * 1000

# Time V2 prediction
t0 = time.perf_counter()
for _ in range(10):
    predictor_v2.predict_candidate_qubits(circ_medium_ir, backend_props)
v2_time = (time.perf_counter() - t0) / 10 * 1000

# Time full compilation
compiler_bench = QBCompiler.from_backend("ibm_fez")
t0 = time.perf_counter()
for _ in range(5):
    compiler_bench.compile(circ_medium)
compile_time = (time.perf_counter() - t0) / 5 * 1000

print(f"\nTiming (averaged):")
print(f"  V1 prediction:    {ml_time:.1f} ms")
print(f"  V2 prediction:    {v2_time:.1f} ms")
print(f"  Full compilation: {compile_time:.1f} ms")
print(f"  V1 speedup:       {compile_time / ml_time:.1f}x faster than compile")

## Summary

| Feature | V1 (XGBoost) | V2 (XGBoost) | GNN (PyTorch) |
|---------|-------------|-------------|---------------|
| Prediction target | Per-qubit probability | Post-routing 2Q count | Per-qubit score |
| `score_layout()` | No | Yes | No |
| `score_layouts()` | No | Yes (batched) | No |
| Graph structure | Flat features | Flat features | Full topology |
| Best for | Fast filtering | Layout comparison | Complex topologies |
| Install extra | `qb-compiler[ml]` | `qb-compiler[ml]` | `qb-compiler[gnn]` |

All three are optional --- the compiler works without them using heuristic
layout selection.  For production workloads on large backends, V1 or V2
is recommended.  The GNN is most useful when topology structure matters
(e.g., IBM heavy-hex at 156 qubits).